In [0]:
from pyspark.sql import functions as F
import pandas as pd

CIDADES_ABRANGENCIA = {
    "RO": ["PIMENTA BUENO","ALTA FLORESTA D'OESTE","SAO FELIPE D'OESTE","ESPIGAO D'OESTE",
           "SERINGUEIRAS","ALTO ALEGRE DOS PARECIS","CHUPINGUAIA","PARECIS","ROLIM DE MOURA",
           "SAO FRANCISCO DO GUAPORE","SAO MIGUEL DO GUAPORE", "ALVORADA D'OESTE","SANTA LUZIA D'OESTE",
           "CACOAL","NOVA BRASILANDIA D'OESTE","COSTA MARQUES","SAO DOMINGOS DO GUAPORE","NOVO HORIZONTE DO OESTE",
           "MINISTRO ANDREAZZA","PRIMAVERA DE RONDONIA", "MIGRANTINOPOLIS", "NOVO PLANO"],
    "MT": ["CASTANHEIRAS","COLNIZA","ARIPUANA","JUARA","JURUENA","COTRIGUACU","RONDOLANDIA",
           "NOVA BANDEIRANTES","NOVA MONTE VERDE","APIACAS","TABAPORA","PORTO DOS GAUCHOS",
           "NOVO HORIZONTE DO NORTE","NOVA MARINGA"],
    "AM": ["MANICORE","APUI"]
}

def adicionar_abrangencia(df):
    df = df.withColumn("uf", F.upper(F.col("uf"))).withColumn("municipio", F.upper(F.col("municipio")))
    
    condicao_abrangencia = F.lit(False)
    for uf, cidades in CIDADES_ABRANGENCIA.items():
        condicao_abrangencia = condicao_abrangencia | ((F.col("uf") == uf) & (F.col("municipio").isin(cidades)))
        
    return df.withColumn("abrangencia", F.when(condicao_abrangencia, F.lit("SIM")).otherwise(F.lit("NAO")))

df_coop_silver = spark.table("silver_coop")

dados = {
    'COOP CNPJ': ['08044854', '02015588', '03632872'],
    'Sigla Coop': ['SICOOB CENTRO', 'SICOOB CREDIP', 'SICOOB CREDISUL'],
    'UF': ['RO', 'RO', 'RO'],
    'MUNICIPIO': ['JI-PARANA', 'PIMENTA BUENO', 'VILHENA']
}

df_filtro = spark.createDataFrame(pd.DataFrame(dados))

df_ativos = (
    df_coop_silver
    .filter(F.col("indicador").isin("ATIVO", "COMPENSACAO"))
    .groupBy("datamovimento", "CNPJ")
    .pivot("indicador", ["ATIVO", "COMPENSACAO"])
    .agg(F.sum("valor"))
    .fillna(0.0)
    .withColumn("valor", F.col("ATIVO") - F.col("COMPENSACAO"))
    .withColumn("indicador", F.lit("ATIVOS"))
    .select("datamovimento", "CNPJ", "indicador", "valor")
)

df_coop_geral = df_coop_silver.filter(~F.col("indicador").isin("ATIVO", "COMPENSACAO")).union(df_ativos)

df_coop_gold = (
    df_coop_geral.alias("c")
    .join(df_filtro.alias("f"), F.col("c.CNPJ") == F.col("f.`COOP CNPJ`"), "inner")
    .select(
        F.lit("balancete").alias("origem"),
        F.col("c.datamovimento"),
        F.col("f.`Sigla Coop`").alias("instituicao"),
        F.col("f.UF").alias("uf"),
        F.col("f.MUNICIPIO").alias("municipio"),
        F.col("c.indicador"),
        F.col("c.valor")
    )
)

df_estban_gold = (
    spark.table("silver_estban")
    .filter(F.col("UF").isin("MT", "RO", "AM"))
    .groupBy("datamovimento", "NOME_INSTITUICAO", "UF", "MUNICIPIO", "indicador")
    .agg((F.sum("valor") / 10.0).alias("valor"))
    .select(
        F.lit("estban").alias("origem"),
        F.col("datamovimento"),
        F.col("NOME_INSTITUICAO").alias("instituicao"),
        F.col("UF").alias("uf"),
        F.col("MUNICIPIO").alias("municipio"),
        F.col("indicador"),
        F.col("valor")
    )
)

df_unificado = df_coop_gold.union(df_estban_gold)
df_unificado = adicionar_abrangencia(df_unificado)

regras_compostos = {
    "DEPOSITOS_TOTAIS": ["DEPOSITO_A_PRAZO", "DEPOSITO_A_VISTA"],
    "CREDITOS_TOTAIS": ["REPASSES + BNDES + FUNDOS", "CREDITO_COMERCIAL"],
    "DEPOSITOS_TOTAIS + POUPANCA": ["DEPOSITO_A_PRAZO", "DEPOSITO_A_VISTA", "POUPANCA"]
}

cols_grupo = ["origem", "datamovimento", "instituicao", "uf", "municipio", "abrangencia"]
dfs_compostos = []

for novo_ind, base_inds in regras_compostos.items():
    df_agg = (
        df_unificado
        .filter(F.col("indicador").isin(base_inds))
        .groupBy(cols_grupo)
        .agg(F.sum("valor").alias("valor"))
        .withColumn("indicador", F.lit(novo_ind))
    )
    dfs_compostos.append(df_agg)

df_final_gold = df_unificado
for df_comp in dfs_compostos:
    df_final_gold = df_final_gold.unionByName(df_comp)

colunas_finais = ["origem", "datamovimento", "instituicao", "uf", "municipio", "indicador", "abrangencia", "valor"]
df_final_gold = df_final_gold.select(*colunas_finais)

(df_final_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_base_consolidada"))

print("Tabela gold_base_consolidada atualizada com sucesso no Databricks!")